# Exploratory Data Analysis (EDA) - Myfitness Dataset
**Author:** Member B (ML Engineering)  
**Purpose:** Analysis of synthetic user demographics, daily behavioral logs, exercise/food interaction distributions, correlations, and outliers.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
DATA_DIR = Path("../../data") if Path("../../data").exists() else Path("data")

## 1. Load Data

In [ ]:
users_df = pd.read_csv(DATA_DIR / "users.csv")
logs_df = pd.read_csv(DATA_DIR / "daily_logs.csv")
exercises_df = pd.read_csv(DATA_DIR / "exercises.csv")
foods_df = pd.read_csv(DATA_DIR / "foods.csv")
interactions_df = pd.read_csv(DATA_DIR / "interactions.csv")

print(
    f"Users: {len(users_df)}, Logs: {len(logs_df)}, Exercises: {len(exercises_df)}, Foods: {len(foods_df)}, Interactions: {len(interactions_df)}"
)

## 2. Demographic Distributions & BMI

In [ ]:
users_df["bmi"] = users_df["weight_kg"] / ((users_df["height_cm"] / 100.0) ** 2)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(users_df["age"], kde=True, ax=axes[0], color="royalblue")
axes[0].set_title("Age Distribution")

sns.histplot(users_df["bmi"], kde=True, ax=axes[1], color="crimson")
axes[1].set_title("BMI Distribution")

users_df["diet_type"].value_counts().plot.pie(
    autopct="%1.1f%%", ax=axes[2], colors=["#66b3ff", "#99ff99"]
)
axes[2].set_title("Diet Preference Ratio (Veg vs Non-Veg)")
plt.tight_layout()
plt.show()

## 3. Daily Habits: Sleep, Water, and Calorie Balance

In [ ]:
logs_df["calorie_balance"] = logs_df["kcal_in"] - logs_df["kcal_out"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(y=logs_df["sleep_hours"], ax=axes[0], color="lightblue")
axes[0].set_title("Daily Sleep Hours")

sns.boxplot(y=logs_df["water_ml"], ax=axes[1], color="lightgreen")
axes[1].set_title("Daily Water Intake (ml)")

sns.histplot(logs_df["calorie_balance"], kde=True, ax=axes[2], color="coral")
axes[2].set_title("Daily Caloric Balance (kcal_in - kcal_out)")
plt.tight_layout()
plt.show()

## 4. Correlation Analysis: Physical Progress vs Daily Habits

In [ ]:
logs_sorted = logs_df.sort_values(["user_id", "date"])
first_w = logs_sorted.groupby("user_id")["weight_kg"].first()
last_w = logs_sorted.groupby("user_id")["weight_kg"].last()
delta_w = (last_w - first_w).rename("net_weight_change_kg")

user_summary = (
    logs_sorted.groupby("user_id")
    .agg(
        {
            "sleep_hours": "mean",
            "water_ml": "mean",
            "calorie_balance": "mean",
            "kcal_in": "mean",
            "kcal_out": "mean",
        }
    )
    .join(delta_w)
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    user_summary.corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f"
)
plt.title("Correlation Matrix: Net Weight Change vs Daily Habits")
plt.show()

## 5. Outlier Detection

In [ ]:
weight_change_outliers = user_summary[
    np.abs(user_summary["net_weight_change_kg"]) > 5.0
]
print(
    f"Users with > 5kg weight change across observation window: {len(weight_change_outliers)}"
)
print(weight_change_outliers.head())